# 🔬 การเปรียบเทียบภาพ Chest X-ray ปอดปกติ (Normal) vs ภาวะ Infiltration ด้วย Grad-CAM

**วัตถุประสงค์ของสมุดโน้ตบุ๊กนี้:**
1. ทำการศึกษาเปรียบเทียบเชิงอธิบายผล (Explainable AI - XAI) ระหว่างภาพปอดปกติ (**Normal / No Finding**) และภาพที่มีภาวะปอดอักเสบสารแทรกซึม (**Infiltration**)
2. ใช้ชุดข้อมูลคลาสละ **100 ภาพ (รวม 200 ภาพ)** คัดเลือกจากคนไข้ที่ไม่ซ้ำกัน (Patient-level) โดยเคส Infiltration มีพิกัด Bounding Box จริงของแพทย์ประกอบครบถ้วน
3. **ใช้ภาพเอกซเรย์ดิบ (Raw Chest X-ray)** โดยยังไม่ผ่านกระบวนการลดสัญญาณรบกวน (Noise Reduction) เพื่อใช้เป็นฐานอ้างอิง (Baseline Comparison)
4. ใช้อัลกอริทึม **Grad-CAM (Gradient-weighted Class Activation Mapping)** บนโมเดล **ResNet50** เพื่อตรวจดูว่าโมเดลโฟกัสจุดใดในภาพปอดปกติ เทียบกับตำแหน่งรอยโรคจริงในภาพ Infiltration

In [1]:
import os
import sys
import glob
import random
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms

# ตรวจสอบ Path
CURRENT_DIR = Path.cwd()
if (CURRENT_DIR / 'sample_manifest_200.csv').exists():
    BASE_DIR = CURRENT_DIR
    DATASET_DIR = CURRENT_DIR.parent.parent
else:
    BASE_DIR = CURRENT_DIR / 'notebooks' / 'Gradcam'
    DATASET_DIR = CURRENT_DIR

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"Dataset Directory: {DATASET_DIR}")
print("โหลดไลบรารี PyTorch, Torchvision, OpenCV สำเร็จ!")

Device: cpu
Dataset Directory: c:\Users\s0955\Github\Enhancing-Chest-X-RAY-Infiltration-Detection-Using-Noise-Reduction-and-Explainable-AI\notebooks
โหลดไลบรารี PyTorch, Torchvision, OpenCV สำเร็จ!


## 1. โหลดข้อมูลชุดทดลอง 200 ภาพ (Normal 100 ภาพ, Infiltration 100 ภาพ)

In [2]:
# โหลด Manifest รายการภาพ 200 ภาพ
manifest_path = BASE_DIR / 'sample_manifest_200.csv'
df_manifest = pd.read_csv(manifest_path)

# โหลด BBox จริงของแพทย์จาก BBox_List_2017.csv
bbox_path = DATASET_DIR / 'BBox_List_2017.csv'
df_bbox = pd.read_csv(bbox_path).iloc[:, :6]
df_bbox.columns = ['Image Index', 'Finding Label', 'x', 'y', 'w', 'h']

print(f"จำนวนภาพในชุดทดลอง: {len(df_manifest)} ภาพ")
print(df_manifest['Class'].value_counts())
print(f"\nจำนวนภาพ Infiltration ที่มี BBox อ้างอิง: {df_manifest[df_manifest['Class'] == 'Infiltration']['Has_BBox'].sum()} ภาพ")
df_manifest.head(6)

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\s0955\\Github\\Enhancing-Chest-X-RAY-Infiltration-Detection-Using-Noise-Reduction-and-Explainable-AI\\notebooks\\BBox_List_2017.csv'

## 2. นิยามคลาส Grad-CAM สำหรับสถาปัตยกรรม ResNet50

Grad-CAM คำนวณน้ำหนักความสำคัญของ Feature Map ในชั้น Convolution สุดท้าย (`layer4[-1]` ของ ResNet50) โดยใช้ Gradient ของคะแนนคลาสเป้าหมายเทียบกับ Activation Map:

$$\alpha_k^c = \frac{1}{Z} \sum_{i}\sum_{j} \frac{\partial Y^c}{\partial A_{i,j}^k}$$

$$L_{\text{Grad-CAM}}^c = \text{ReLU}\left( \sum_k \alpha_k^c A^k \right)$$

In [ ]:
class ResNet50GradCAM:
    def __init__(self, model, target_layer=None):
        self.model = model.to(device)
        self.model.eval()
        self.target_layer = target_layer if target_layer is not None else self.model.layer4[-1]
        
        self.gradients = []
        self.activations = []
        
        # ติดตั้ง Hooks เพื่อดึง Activation และ Gradient
        self.target_layer.register_forward_hook(self._save_activation)
        self.target_layer.register_full_backward_hook(self._save_gradient)
        
    def _save_activation(self, module, inp, out):
        self.activations.append(out)
        
    def _save_gradient(self, module, grad_inp, grad_out):
        self.gradients.append(grad_out[0])
        
    def generate_heatmap(self, input_tensor, class_idx=None):
        self.gradients.clear()
        self.activations.clear()
        
        # Forward pass
        output = self.model(input_tensor)
        if class_idx is None:
            class_idx = output.argmax(dim=1).item()
            
        # Backward pass หา gradient เทียบกับคลาสเป้าหมาย
        self.model.zero_grad()
        score = output[:, class_idx]
        score.backward(retain_graph=True)
        
        # คำนวณ GAP ของ Gradients
        grad = self.gradients[0][0].detach()
        act = self.activations[0][0].detach()
        weights = torch.mean(grad, dim=(1, 2), keepdim=True)
        
        # รวม Linear Combination แล้วผ่าน ReLU
        cam = torch.sum(weights * act, dim=0).clamp(min=0).cpu().numpy()
        
        # Normalize สู่ช่วง 0 ถึง 1
        if cam.max() > cam.min():
            cam = (cam - cam.min()) / (cam.max() - cam.min())
        else:
            cam = np.zeros_like(cam)
            
        return cam, class_idx

# โหลดโมเดล ResNet50 Pretrained ImageNet
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
gradcam = ResNet50GradCAM(model)

# Pipeline เตรียมภาพเข้าสู่ ResNet50
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print("ตั้งค่าโมเดล ResNet50 และคลาส Grad-CAM พร้อมใช้งานแล้ว!")

## 3. ฟังก์ชันสร้างภาพ Overlay (Grad-CAM + Chest X-ray)

In [ ]:
def overlay_gradcam_on_cxr(img_pil, cam, alpha=0.45, colormap=cv2.COLORMAP_JET):
    """
    นำ Heatmap ของ Grad-CAM มาขยายเท่าขนาดภาพต้นฉบับ แล้วซ้อนทับ (Blend) บนภาพ Chest X-ray
    """
    w, h = img_pil.size
    cam_resized = cv2.resize(cam, (w, h))
    heatmap = np.uint8(255 * cam_resized)
    heatmap_colored = cv2.applyColorMap(heatmap, colormap)
    heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)
    
    cxr_rgb = np.array(img_pil.convert('RGB'))
    blended = np.uint8(alpha * heatmap_colored + (1 - alpha) * cxr_rgb)
    return blended, cam_resized

def process_image(img_path):
    img_pil = Image.open(img_path)
    input_tensor = transform(img_pil.convert('RGB')).unsqueeze(0).to(device)
    cam, class_idx = gradcam.generate_heatmap(input_tensor)
    overlay, cam_full = overlay_gradcam_on_cxr(img_pil, cam)
    return img_pil, cam_full, overlay

## 4. เปรียบเทียบแบบเคสต่อเคส (Pairwise Comparison: Normal vs Infiltration)
แสดงภาพต้นฉบับ, Heatmap, ภาพซ้อนทับ (Grad-CAM + CXR) และกรอบ Bounding Box จริงของแพทย์ (สำหรับเคส Infiltration)

In [ ]:
def compare_normal_vs_infil(normal_idx=0, infil_idx=0):
    normal_row = df_manifest[df_manifest['Class'] == 'Normal'].iloc[normal_idx]
    infil_row = df_manifest[df_manifest['Class'] == 'Infiltration'].iloc[infil_idx]
    
    norm_path = DATASET_DIR / normal_row['Relative_Path']
    infil_path = DATASET_DIR / infil_row['Relative_Path']
    
    # ประมวลผลทั้งสองภาพ
    norm_img, norm_cam, norm_overlay = process_image(norm_path)
    infil_img, infil_cam, infil_overlay = process_image(infil_path)
    
    # ดึง Bounding Box ของเคส Infiltration
    infil_name = infil_row['Image Index']
    boxes = df_bbox[df_bbox['Image Index'] == infil_name]
    
    # พล็อตเปรียบเทียบ 2 แถว 3 คอลัมน์
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # แถวบน: ภาพปอดปกติ (Normal / No Finding)
    axes[0, 0].imshow(norm_img, cmap='gray')
    axes[0, 0].set_title(f"[NORMAL] Raw CXR\n{normal_row['Image Index']}", fontsize=11, weight='bold')
    axes[0, 0].axis('off')
    
    axes[0, 1].imshow(norm_cam, cmap='jet')
    axes[0, 1].set_title("[NORMAL] Grad-CAM Heatmap", fontsize=11, weight='bold')
    axes[0, 1].axis('off')
    
    axes[0, 2].imshow(norm_overlay)
    axes[0, 2].set_title("[NORMAL] Overlay (CXR + Grad-CAM)", fontsize=11, weight='bold')
    axes[0, 2].axis('off')
    
    # แถวล่าง: ภาพปอดอักเสบ (Infiltration)
    axes[1, 0].imshow(infil_img, cmap='gray')
    axes[1, 0].set_title(f"[INFILTRATION] Raw CXR\n{infil_name}", fontsize=11, weight='bold')
    axes[1, 0].axis('off')
    
    axes[1, 1].imshow(infil_cam, cmap='jet')
    axes[1, 1].set_title("[INFILTRATION] Grad-CAM Heatmap", fontsize=11, weight='bold')
    axes[1, 1].axis('off')
    
    # แถวล่างขวา: Overlay พร้อมตีกรอบรอยโรคจริง (Doctor's Ground-Truth BBox)
    axes[1, 2].imshow(infil_overlay)
    for _, b in boxes.iterrows():
        bx, by, bw, bh = float(b['x']), float(b['y']), float(b['w']), float(b['h'])
        rect = patches.Rectangle((bx, by), bw, bh, linewidth=2.5, edgecolor='#00FF66', facecolor='none', linestyle='--')
        axes[1, 2].add_patch(rect)
        axes[1, 2].text(
            bx, max(0, by - 10), "BBox: Infiltrate",
            color='black', fontsize=9, weight='bold',
            bbox=dict(boxstyle='square,pad=0.2', facecolor='#00FF66', edgecolor='none', alpha=0.9)
        )
    axes[1, 2].set_title("[INFILTRATION] Overlay + Doctor's BBox (Green)", fontsize=11, weight='bold')
    axes[1, 2].axis('off')
    
    plt.tight_layout()
    plt.show()

# รันตัวอย่างคู่แรก
compare_normal_vs_infil(normal_idx=0, infil_idx=0)

## 5. แสดงตารางเปรียบเทียบหลายภาพพร้อมกัน (4 Normal vs 4 Infiltration)
ตรวจดูรูปแบบการกระจายตัวของ Heatmap ในเคสปกติหลาย ๆ คนไข้ เทียบกับเคสที่มีภาวะฝ้าแทรกซึม

In [ ]:
def show_batch_comparison(n_samples=4):
    fig, axes = plt.subplots(n_samples, 4, figsize=(18, 4.2 * n_samples))
    
    norm_samples = df_manifest[df_manifest['Class'] == 'Normal'].iloc[:n_samples]
    infil_samples = df_manifest[df_manifest['Class'] == 'Infiltration'].iloc[:n_samples]
    
    for i in range(n_samples):
        # 1. Normal Raw CXR
        n_row = norm_samples.iloc[i]
        n_img, n_cam, n_over = process_image(DATASET_DIR / n_row['Relative_Path'])
        axes[i, 0].imshow(n_img, cmap='gray')
        axes[i, 0].set_title(f"Normal #{i+1}: {n_row['Image Index']}", fontsize=10)
        axes[i, 0].axis('off')
        
        # 2. Normal Overlay
        axes[i, 1].imshow(n_over)
        axes[i, 1].set_title(f"Normal #{i+1} Grad-CAM", fontsize=10)
        axes[i, 1].axis('off')
        
        # 3. Infiltration Raw CXR
        inf_row = infil_samples.iloc[i]
        inf_img, inf_cam, inf_over = process_image(DATASET_DIR / inf_row['Relative_Path'])
        axes[i, 2].imshow(inf_img, cmap='gray')
        axes[i, 2].set_title(f"Infiltration #{i+1}: {inf_row['Image Index']}", fontsize=10)
        axes[i, 2].axis('off')
        
        # 4. Infiltration Overlay + BBox
        axes[i, 3].imshow(inf_over)
        inf_boxes = df_bbox[df_bbox['Image Index'] == inf_row['Image Index']]
        for _, b in inf_boxes.iterrows():
            bx, by, bw, bh = float(b['x']), float(b['y']), float(b['w']), float(b['h'])
            rect = patches.Rectangle((bx, by), bw, bh, linewidth=2, edgecolor='#00FF66', facecolor='none')
            axes[i, 3].add_patch(rect)
        axes[i, 3].set_title(f"Infiltration #{i+1} + BBox", fontsize=10)
        axes[i, 3].axis('off')
        
    plt.tight_layout()
    plt.show()

show_batch_comparison(n_samples=4)

## 6. การวัดความสอดคล้องระหว่าง Grad-CAM และ Ground-Truth Bounding Box (Pointing Game / XAI Alignment)
วัดว่าตำแหน่งที่ Grad-CAM มีค่าความเข้มข้นสูงสุด (Peak Activation) ตกอยู่ภายในกรอบ Bounding Box ของรอยโรคที่แพทย์ระบุไว้หรือไม่

In [ ]:
infil_records = df_manifest[df_manifest['Class'] == 'Infiltration']
hits = 0
total_tested = 0

print("กำลังคำนวณการตรวจจับตำแหน่ง Peak Attention บนภาพ Infiltration 100 ภาพ...")

results_list = []
for idx, row in infil_records.iterrows():
    img_name = row['Image Index']
    img_path = DATASET_DIR / row['Relative_Path']
    boxes = df_bbox[df_bbox['Image Index'] == img_name]
    if boxes.empty:
        continue
        
    img_pil, cam, _ = process_image(img_path)
    
    # หาตำแหน่งพิกัด (y, x) ที่มีค่า Activation สูงสุด
    peak_y, peak_x = np.unravel_index(np.argmax(cam), cam.shape)
    
    # ตรวจสอบว่าจุด Peak อยู่ใน Bounding Box ใดๆ หรือไม่
    is_hit = False
    for _, b in boxes.iterrows():
        bx, by, bw, bh = float(b['x']), float(b['y']), float(b['w']), float(b['h'])
        if (bx <= peak_x <= bx + bw) and (by <= peak_y <= by + bh):
            is_hit = True
            break
            
    if is_hit:
        hits += 1
    total_tested += 1
    results_list.append({
        'Image Index': img_name,
        'Peak_X': peak_x,
        'Peak_Y': peak_y,
        'Inside_BBox': is_hit
    })

df_xai_eval = pd.DataFrame(results_list)
hit_rate = (hits / total_tested) * 100
print(f"\nผลการประเมิน Pointing Game บนภาพ Infiltration ({total_tested} ภาพ):")
print(f"- จุด Peak Activation ตกใน Bounding Box จริง: {hits}/{total_tested} ({hit_rate:.1f}%)")
print("หมายเหตุ: โมเดล Baseline Pretrained จาก ImageNet ยังไม่ได้ Fine-tune ให้เฉพาะเจาะจงกับฝ้า Infiltration จึงทำหน้าที่เป็น Base Comparator สำหรับรอบถัดไป")
df_xai_eval.head(10)

## 7. การประเมินผลด้วย Confusion Matrix ทั้ง 2 รูปแบบ
การประเมินผลลัพธ์ของโมเดลแบ่งออกเป็น 2 มุมมองหลัก:
1. **Classification Confusion Matrix:** วัดผลการจำแนกประเภทโรค (Normal vs Infiltration)
2. **XAI Localization Confusion Matrix:** วัดความแม่นยำในการชี้ตำแหน่ง (Pointing Game Hit vs Miss เทียบกับ BBox ของแพทย์)

In [ ]:
clf_cm_path = BASE_DIR / 'output' / 'confusion_matrix_classification.png'
loc_cm_path = BASE_DIR / 'output' / 'confusion_matrix_xai_localization.png'
summary_csv = BASE_DIR / 'reports' / 'gradcam_confusion_matrix_summary.csv'

if summary_csv.exists():
    display(pd.read_csv(summary_csv))

fig, axes = plt.subplots(1, 2, figsize=(15, 6), dpi=150)
if clf_cm_path.exists():
    axes[0].imshow(Image.open(clf_cm_path))
    axes[0].axis('off')
    axes[0].set_title('1. Classification Matrix', fontsize=12, weight='bold')

if loc_cm_path.exists():
    axes[1].imshow(Image.open(loc_cm_path))
    axes[1].axis('off')
    axes[1].set_title('2. XAI Localization Matrix (Pointing Game)', fontsize=12, weight='bold')

plt.tight_layout()
plt.show()

## 8. ฟังก์ชันสุ่มดูคู่ภาพแบบ Interactive (Random Pair Explorer)
รันเซลล์นี้เพื่อสุ่มดูคู่ภาพ Normal vs Infiltration คู่ใหม่ ๆ จากชุดทดลอง 200 ภาพ

In [ ]:
def explore_random_pair():
    rand_norm = random.randint(0, 99)
    rand_infil = random.randint(0, 99)
    print(f"สุ่มคู่ที่: Normal #{rand_norm} | Infiltration #{rand_infil}")
    compare_normal_vs_infil(rand_norm, rand_infil)

explore_random_pair()